# Scrapper

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import pandas as pd
from datetime import datetime
import re

# Configurar opciones de Selenium
options = Options()
options.add_argument("--headless")  # Puedes activar modo headless si lo deseas
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

# Configuraciones adicionales para evitar detección
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins-discovery")
options.add_argument("--disable-default-apps")

def cargar_datos_desde_excel(ruta_archivo):
    """
    Carga marca, color, capacidad y SKU real desde un archivo Excel que debe tener las columnas:
    - SKU_URL: Código SKU extraído de la URL del producto
    - SKU_REAL: Código SKU real del producto
    - Color: Color del celular
    - Marca: Marca del celular
    - Capacidad: Capacidad del celular
    """
    try:
        df_datos = pd.read_excel(ruta_archivo, sheet_name="Sears")
        
        # Verificar que las columnas necesarias existan
        required_columns = ['Sku']
        missing_columns = [col for col in required_columns if col not in df_datos.columns]
        if missing_columns:
            print(f"Error: El archivo Excel debe tener al menos la columna: {missing_columns}")
            return {}, {}, {}, {}
        
        # Convertir a string para evitar problemas de tipos
        df_datos['Sku'] = df_datos['Sku'].astype(str)
        
        # Limpiar valores NaN o vacíos
        df_datos = df_datos.dropna(subset=['Sku'])
        df_datos = df_datos[df_datos['Sku'].str.strip() != '']
        
        # Crear diccionarios SKU_URL -> Datos
        sku_marca_dict = {}
        sku_color_dict = {}
        sku_capacidad_dict = {}
        sku_real_dict = {}  # Nuevo diccionario para mapear SKU_URL -> SKU_REAL
        
        for _, row in df_datos.iterrows():
            sku_url = row['Sku']  # SKU extraído de la URL
            
            # Solo agregar si la columna existe y tiene valor
            if 'Marca' in df_datos.columns and pd.notna(row['Marca']) and str(row['Marca']).strip() != '':
                sku_marca_dict[sku_url] = str(row['Marca']).strip()
            
            if 'Color' in df_datos.columns and pd.notna(row['Color']) and str(row['Color']).strip() != '':
                sku_color_dict[sku_url] = str(row['Color']).strip()
            
            if 'Capacidad' in df_datos.columns and pd.notna(row['Capacidad']) and str(row['Capacidad']).strip() != '':
                sku_capacidad_dict[sku_url] = str(row['Capacidad']).strip()
            
            # Mapear SKU_URL -> SKU_REAL si existe la columna
            if 'Sku_Real' in df_datos.columns and pd.notna(row['SKU_Real']) and str(row['SKU_Real']).strip() != '':
                sku_real_dict[sku_url] = str(row['Sku_Real']).strip()
        
        print(f"Datos cargados exitosamente desde 'diccionario.xlsx' hoja 'Sears':")
        print(f"- Marcas: {len(sku_marca_dict)} registros")
        print(f"- Colores: {len(sku_color_dict)} registros")
        print(f"- Capacidades: {len(sku_capacidad_dict)} registros")
        print(f"- SKUs reales: {len(sku_real_dict)} registros")
        return sku_marca_dict, sku_color_dict, sku_capacidad_dict, sku_real_dict
        
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {ruta_archivo}")
        print("Asegúrate de que el archivo 'diccionario.xlsx' esté en la carpeta correcta")
        return {}, {}, {}, {}
    except Exception as e:
        print(f"Error al cargar el archivo de datos: {e}")
        return {}, {}, {}, {}

# Cargar datos al inicio del script
ruta_archivo_datos = r"c:\Users\c50012218\Documents\diccionario.xlsx"
sku_marca_dict, sku_color_dict, sku_capacidad_dict, sku_real_dict = cargar_datos_desde_excel(ruta_archivo_datos)

def extraer_info_producto_sears(product, fecha_consulta):
    """
    Extrae información de cada producto de Sears
    """
    try:
        # Nombre del producto - desde el h3
        nombre = ""
        nombre_tag = product.find("h3", class_="CardProduct_h4__C5Mg2")
        if nombre_tag:
            nombre = nombre_tag.get_text(strip=True)
        
        # SKU - extraer del href primero para usarlo en la búsqueda, luego buscar SKU real
        sku_url = ""
        sku_real = ""
        url_producto = ""
        if product.name == 'a' and product.get("href"):
            href = product["href"]
            if href.startswith("/"):
                url_producto = "https://www.sears.com.mx" + href
            elif href.startswith("https://"):
                url_producto = href
            
            # Extraer SKU del URL
            sku_match = re.search(r'/producto/(\d+)/', href)
            if sku_match:
                sku_url = sku_match.group(1)
                
                # Buscar SKU real en el diccionario
                if sku_url in sku_real_dict:
                    sku_real = sku_real_dict[sku_url]
                else:
                    sku_real = sku_url  # Si no hay mapeo, usar el SKU de la URL
        
        # MARCA - SOLO del Excel, si no está, dejar vacía
        marca = ""
        if sku_url and sku_url in sku_marca_dict:
            marca = sku_marca_dict[sku_url]
        
        # COLOR - Primero del nombre, luego del Excel
        color = ""
        if nombre:
            nombre_lower = nombre.lower()
            colores = [
                'titanio negro', 'titanio blanco', 'titanio del desierto', 'titanio natural', 'negro',
                'black', 'blanco', 'white', 'azul', 'blue', 'rojo', 'red',
                'verde', 'green', 'amarillo', 'yellow', 'rosa', 'pink', 'morado', 'purple', 'purpura',
                'gris', 'gray', 'grey', 'dorado', 'gold', 'plateado', 'silver', 'plata', 'cian',
                'lila', 'grafito', 'purpura', 'lavanda', 'ultramarino', 'medianoche', 'cafe',
                'menta', 'multicolor', 'beige', 'violeta', 'naranja', 'midnight', 'titanio', 'starlight'
            ]
            
            for c in colores:
                if c in nombre_lower:
                    color = c.capitalize()
                    break
        
        # Si no se encontró color en el nombre, buscar en Excel
        if not color and sku_url and sku_url in sku_color_dict:
            color = sku_color_dict[sku_url]
        
        # CAPACIDAD - SOLO del Excel, si no está, dejar vacía
        capacidad = ""
        if sku_url and sku_url in sku_capacidad_dict:
            capacidad = sku_capacidad_dict[sku_url]
        
        # Función para limpiar precios
        def limpiar_precio_texto(precio_texto):
            if precio_texto:
                precio_limpio = re.sub(r'[^\d,.]', '', precio_texto.replace('MXN', '').replace('$', ''))
                return precio_limpio.strip()
            return ""
        
        # Precios
        precio_original = ""
        precio_descuento = ""
        
        precio_orig_tag = product.find("span", class_="CardProduct_textUnderline___Yf8H")
        if precio_orig_tag:
            precio_original = limpiar_precio_texto(precio_orig_tag.get_text(strip=True))
        
        precio_desc_tag = product.find("p", class_="CardProduct_precio1__5GHph")
        if precio_desc_tag:
            precio_descuento = limpiar_precio_texto(precio_desc_tag.get_text(strip=True))
        
        if not precio_original and precio_descuento:
            precio_original = precio_descuento
            precio_descuento = "No discount"
        elif precio_original and not precio_descuento:
            precio_descuento = "No discount"
        elif not precio_original and not precio_descuento:
            precio_original = "No inventory"
            precio_descuento = "No discount"
        
        def limpiar_precio_calculo(precio):
            if precio and precio not in ["No discount", "No inventory"]:
                cleaned = re.sub(r'[^\d.]', '', precio.replace(',', ''))
                try:
                    return float(cleaned)
                except:
                    return None
            return None
        
        p_orig = limpiar_precio_calculo(precio_original)
        p_desc = limpiar_precio_calculo(precio_descuento)
        
        if p_orig and p_desc and p_orig > p_desc:
            porcentaje_descuento = round(100 * (p_orig - p_desc) / p_orig, 2)
            porcentaje_str = f"{porcentaje_descuento}%"
        else:
            porcentaje_str = "No discount"
        
        return {
            "Marca": marca,
            "Nombre": nombre,
            "Color": color,
            "Capacidad": capacidad,
            "SKU": sku_real,  # Usar el SKU real en lugar del SKU de la URL
            "SKU_URL": sku_url,  # Mantener también el SKU de la URL para referencia
            "URL": url_producto,
            "FechaConsulta": fecha_consulta,
            "Precio_original": precio_original,
            "Precio_descuento": precio_descuento,
            "Porcentaje_descuento": porcentaje_str
        }
        
    except Exception as e:
        print(f"Error al extraer producto: {e}")
        return None

def scrape_sears_smartphones():
    """
    Función principal para hacer scraping de smartphones en Sears
    """
    base_url = "https://www.sears.com.mx/cat/telefon%C3%ADa/celulares?id=16108&page="
    print("Iniciando scraping de Sears...")
    
    driver = webdriver.Chrome(options=options)
    data = []
    fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    try:
        for page in range(1, 6):  # Iterar por las páginas 1 a 5
            url = base_url + str(page)
            print(f"Procesando página: {page}")
            driver.get(url)
            time.sleep(30)  # Esperar a que cargue la página
            
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(15)
            
            soup = BeautifulSoup(driver.page_source, "html.parser")
            products = soup.find_all("a", href=re.compile(r"/producto/"))
            
            print(f"Productos encontrados en página {page}: {len(products)}")
            
            for product in products:
                info = extraer_info_producto_sears(product, fecha_consulta)
                if info and (info["Nombre"] or info["URL"]):
                    data.append(info)
        
        print(f"Total de productos extraídos: {len(data)}")
        
    except Exception as e:
        print(f"Error durante el scraping: {e}")
        import traceback
        traceback.print_exc()
    finally:
        driver.quit()
    
    return data

# Ejecutar el scraping
resultados_sears = scrape_sears_smartphones()

# Exportar a CSV
if resultados_sears:
    df_sears = pd.DataFrame(resultados_sears)
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"sears_smartphones_{fecha_hoy}_p1.csv"
    df_sears.to_csv(nombre_archivo, index=False, encoding="utf-8")
    print(f"Datos exportados a {nombre_archivo}")
    print(f"Total de productos: {len(df_sears)}")
    print("\nPrimeros 5 productos:")
    print(df_sears.head())
else:
    print("No se encontraron productos.")

Datos cargados exitosamente desde 'diccionario.xlsx' hoja 'Sears':
- Marcas: 5863 registros
- Colores: 5583 registros
- Capacidades: 5732 registros
- SKUs reales: 0 registros
Iniciando scraping de Sears...
Procesando página: 1
Productos encontrados en página 1: 100
Procesando página: 2
Productos encontrados en página 2: 100
Procesando página: 3
Productos encontrados en página 3: 100
Procesando página: 4
Productos encontrados en página 4: 100
Procesando página: 5
Productos encontrados en página 5: 100
Total de productos extraídos: 500
Datos exportados a sears_smartphones_2026-05-14_p1.csv
Total de productos: 500

Primeros 5 productos:
    Marca                                             Nombre    Color  \
0   APPLE          Celular Iphone 17 256Gb Negro R9 (Telcel)    Negro   
1   APPLE  APPLE IPHONE 12 PRO MAX 256GB AZUL REACONDICIO...     Azul   
2   APPLE       Celular Iphone 17 Air 256Gb Azul R9 (Telcel)     Azul   
3   APPLE  Apple iPhone 12 Pro Max 256gb Grafito Reacondi...  Graf

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import pandas as pd
from datetime import datetime
import re

# Configurar opciones de Selenium
options = Options()
options.add_argument("--headless")  # Puedes activar modo headless si lo deseas
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

# Configuraciones adicionales para evitar detección
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins-discovery")
options.add_argument("--disable-default-apps")

def cargar_datos_desde_excel(ruta_archivo):
    """
    Carga marca, color, capacidad y SKU real desde un archivo Excel que debe tener las columnas:
    - SKU_URL: Código SKU extraído de la URL del producto
    - SKU_REAL: Código SKU real del producto
    - Color: Color del celular
    - Marca: Marca del celular
    - Capacidad: Capacidad del celular
    """
    try:
        df_datos = pd.read_excel(ruta_archivo, sheet_name="Sears")
        
        # Verificar que las columnas necesarias existan
        required_columns = ['Sku']
        missing_columns = [col for col in required_columns if col not in df_datos.columns]
        if missing_columns:
            print(f"Error: El archivo Excel debe tener al menos la columna: {missing_columns}")
            return {}, {}, {}, {}
        
        # Convertir a string para evitar problemas de tipos
        df_datos['Sku'] = df_datos['Sku'].astype(str)
        
        # Limpiar valores NaN o vacíos
        df_datos = df_datos.dropna(subset=['Sku'])
        df_datos = df_datos[df_datos['Sku'].str.strip() != '']
        
        # Crear diccionarios SKU_URL -> Datos
        sku_marca_dict = {}
        sku_color_dict = {}
        sku_capacidad_dict = {}
        sku_real_dict = {}  # Nuevo diccionario para mapear SKU_URL -> SKU_REAL
        
        for _, row in df_datos.iterrows():
            sku_url = row['Sku']  # SKU extraído de la URL
            
            # Solo agregar si la columna existe y tiene valor
            if 'Marca' in df_datos.columns and pd.notna(row['Marca']) and str(row['Marca']).strip() != '':
                sku_marca_dict[sku_url] = str(row['Marca']).strip()
            
            if 'Color' in df_datos.columns and pd.notna(row['Color']) and str(row['Color']).strip() != '':
                sku_color_dict[sku_url] = str(row['Color']).strip()
            
            if 'Capacidad' in df_datos.columns and pd.notna(row['Capacidad']) and str(row['Capacidad']).strip() != '':
                sku_capacidad_dict[sku_url] = str(row['Capacidad']).strip()
            
            # Mapear SKU_URL -> SKU_REAL si existe la columna
            if 'Sku_Real' in df_datos.columns and pd.notna(row['Sku_Real']) and str(row['Sku_Real']).strip() != '':
                sku_real_dict[sku_url] = str(row['Sku_Real']).strip()
        
        print(f"Datos cargados exitosamente desde 'diccionario.xlsx' hoja 'Sears':")
        print(f"- Marcas: {len(sku_marca_dict)} registros")
        print(f"- Colores: {len(sku_color_dict)} registros")
        print(f"- Capacidades: {len(sku_capacidad_dict)} registros")
        print(f"- SKUs reales: {len(sku_real_dict)} registros")
        return sku_marca_dict, sku_color_dict, sku_capacidad_dict, sku_real_dict
        
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {ruta_archivo}")
        print("Asegúrate de que el archivo 'diccionario.xlsx' esté en la carpeta correcta")
        return {}, {}, {}, {}
    except Exception as e:
        print(f"Error al cargar el archivo de datos: {e}")
        return {}, {}, {}, {}

# Cargar datos al inicio del script
ruta_archivo_datos = r"c:\Users\c50012218\Documents\diccionario.xlsx"
sku_marca_dict, sku_color_dict, sku_capacidad_dict, sku_real_dict = cargar_datos_desde_excel(ruta_archivo_datos)

def extraer_info_producto_sears(product, fecha_consulta):
    """
    Extrae información de cada producto de Sears
    """
    try:
        # Nombre del producto - desde el h3
        nombre = ""
        nombre_tag = product.find("h3", class_="CardProduct_h4__C5Mg2")
        if nombre_tag:
            nombre = nombre_tag.get_text(strip=True)
        
        # SKU - extraer del href primero para usarlo en la búsqueda, luego buscar SKU real
        sku_url = ""
        sku_real = ""
        url_producto = ""
        if product.name == 'a' and product.get("href"):
            href = product["href"]
            if href.startswith("/"):
                url_producto = "https://www.sears.com.mx" + href
            elif href.startswith("https://"):
                url_producto = href
            
            # Extraer SKU del URL
            sku_match = re.search(r'/producto/(\d+)/', href)
            if sku_match:
                sku_url = sku_match.group(1)
                
                # Buscar SKU real en el diccionario
                if sku_url in sku_real_dict:
                    sku_real = sku_real_dict[sku_url]
                else:
                    sku_real = sku_url  # Si no hay mapeo, usar el SKU de la URL
        
        # MARCA - SOLO del Excel, si no está, dejar vacía
        marca = ""
        if sku_url and sku_url in sku_marca_dict:
            marca = sku_marca_dict[sku_url]
        
        # COLOR - Primero del nombre, luego del Excel
        color = ""
        if nombre:
            nombre_lower = nombre.lower()
            colores = [
                'titanio negro', 'titanio blanco', 'titanio del desierto', 'titanio natural', 'negro',
                'black', 'blanco', 'white', 'azul', 'blue', 'rojo', 'red',
                'verde', 'green', 'amarillo', 'yellow', 'rosa', 'pink', 'morado', 'purple', 'purpura',
                'gris', 'gray', 'grey', 'dorado', 'gold', 'plateado', 'silver', 'plata', 'cian',
                'lila', 'grafito', 'purpura', 'lavanda', 'ultramarino', 'medianoche', 'cafe',
                'menta', 'multicolor', 'beige', 'violeta', 'naranja', 'midnight', 'titanio', 'starlight'
            ]
            
            for c in colores:
                if c in nombre_lower:
                    color = c.capitalize()
                    break
        
        # Si no se encontró color en el nombre, buscar en Excel
        if not color and sku_url and sku_url in sku_color_dict:
            color = sku_color_dict[sku_url]
        
        # CAPACIDAD - SOLO del Excel, si no está, dejar vacía
        capacidad = ""
        if sku_url and sku_url in sku_capacidad_dict:
            capacidad = sku_capacidad_dict[sku_url]
        
        # Función para limpiar precios
        def limpiar_precio_texto(precio_texto):
            if precio_texto:
                precio_limpio = re.sub(r'[^\d,.]', '', precio_texto.replace('MXN', '').replace('$', ''))
                return precio_limpio.strip()
            return ""
        
        # Precios
        precio_original = ""
        precio_descuento = ""
        
        precio_orig_tag = product.find("span", class_="CardProduct_textUnderline___Yf8H")
        if precio_orig_tag:
            precio_original = limpiar_precio_texto(precio_orig_tag.get_text(strip=True))
        
        precio_desc_tag = product.find("p", class_="CardProduct_precio1__5GHph")
        if precio_desc_tag:
            precio_descuento = limpiar_precio_texto(precio_desc_tag.get_text(strip=True))
        
        if not precio_original and precio_descuento:
            precio_original = precio_descuento
            precio_descuento = "No discount"
        elif precio_original and not precio_descuento:
            precio_descuento = "No discount"
        elif not precio_original and not precio_descuento:
            precio_original = "No inventory"
            precio_descuento = "No discount"
        
        def limpiar_precio_calculo(precio):
            if precio and precio not in ["No discount", "No inventory"]:
                cleaned = re.sub(r'[^\d.]', '', precio.replace(',', ''))
                try:
                    return float(cleaned)
                except:
                    return None
            return None
        
        p_orig = limpiar_precio_calculo(precio_original)
        p_desc = limpiar_precio_calculo(precio_descuento)
        
        if p_orig and p_desc and p_orig > p_desc:
            porcentaje_descuento = round(100 * (p_orig - p_desc) / p_orig, 2)
            porcentaje_str = f"{porcentaje_descuento}%"
        else:
            porcentaje_str = "No discount"
        
        return {
            "Marca": marca,
            "Nombre": nombre,
            "Color": color,
            "Capacidad": capacidad,
            "SKU": sku_real,  # Usar el SKU real en lugar del SKU de la URL
            "SKU_URL": sku_url,  # Mantener también el SKU de la URL para referencia
            "URL": url_producto,
            "FechaConsulta": fecha_consulta,
            "Precio_original": precio_original,
            "Precio_descuento": precio_descuento,
            "Porcentaje_descuento": porcentaje_str
        }
        
    except Exception as e:
        print(f"Error al extraer producto: {e}")
        return None

def scrape_sears_smartphones():
    """
    Función principal para hacer scraping de smartphones en Sears
    """
    base_url = "https://www.sears.com.mx/cat/telefon%C3%ADa/celulares?id=16108&page="
    print("Iniciando scraping de Sears...")
    
    driver = webdriver.Chrome(options=options)
    data = []
    fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    try:
        for page in range(6, 11):  # Iterar por las páginas 6 a 10
            url = base_url + str(page)
            print(f"Procesando página: {page}")
            driver.get(url)
            time.sleep(30)  # Esperar a que cargue la página
            
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(15)
            
            soup = BeautifulSoup(driver.page_source, "html.parser")
            products = soup.find_all("a", href=re.compile(r"/producto/"))
            
            print(f"Productos encontrados en página {page}: {len(products)}")
            
            for product in products:
                info = extraer_info_producto_sears(product, fecha_consulta)
                if info and (info["Nombre"] or info["URL"]):
                    data.append(info)
        
        print(f"Total de productos extraídos: {len(data)}")
        
    except Exception as e:
        print(f"Error durante el scraping: {e}")
        import traceback
        traceback.print_exc()
    finally:
        driver.quit()
    
    return data

# Ejecutar el scraping
resultados_sears = scrape_sears_smartphones()

# Exportar a CSV
if resultados_sears:
    df_sears = pd.DataFrame(resultados_sears)
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"sears_smartphones_{fecha_hoy}_p2.csv"
    df_sears.to_csv(nombre_archivo, index=False, encoding="utf-8")
    print(f"Datos exportados a {nombre_archivo}")
    print(f"Total de productos: {len(df_sears)}")
    print("\nPrimeros 5 productos:")
    print(df_sears.head())
else:
    print("No se encontraron productos.")

Datos cargados exitosamente desde 'diccionario.xlsx' hoja 'Sears':
- Marcas: 5863 registros
- Colores: 5583 registros
- Capacidades: 5732 registros
- SKUs reales: 0 registros
Iniciando scraping de Sears...
Procesando página: 6
Productos encontrados en página 6: 100
Procesando página: 7
Productos encontrados en página 7: 100
Procesando página: 8
Productos encontrados en página 8: 100
Procesando página: 9
Productos encontrados en página 9: 100
Procesando página: 10
Productos encontrados en página 10: 100
Total de productos extraídos: 500
Datos exportados a sears_smartphones_2026-05-14_p2.csv
Total de productos: 500

Primeros 5 productos:
      Marca                                             Nombre    Color  \
0  MOTOROLA     Motorola G06 256Gb 4Ram Color Azul Nuevo Libre     Azul   
1      OPPO  Celular Oppo Find X9 Pro 512 5G Titan R9 (Telcel)  Titanio   
2      OPPO  Celular Oppo Find X9 Pro 512 5G Blanc R9 (Telcel)   Blanco   
3      OPPO    Celular Oppo Find X9 512Gb 5G Negro R9 (T

In [3]:
import pandas as pd
from datetime import datetime

def unir_archivos_sears_p1_p2():
    """
    Une específicamente los archivos p1 y p2 de Sears
    """
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    
    # Nombres de los archivos a combinar
    archivo_p1 = f"sears_smartphones_{fecha_hoy}_p1.csv"
    archivo_p2 = f"sears_smartphones_{fecha_hoy}_p2.csv"
    
    dataframes = []
    
    # Leer archivo p1
    try:
        df_p1 = pd.read_csv(archivo_p1, encoding='utf-8')
        print(f"Archivo P1 cargado: {len(df_p1)} registros")
        dataframes.append(df_p1)
    except FileNotFoundError:
        print(f"No se encontró el archivo: {archivo_p1}")
    except Exception as e:
        print(f"Error al leer {archivo_p1}: {e}")
    
    # Leer archivo p2
    try:
        df_p2 = pd.read_csv(archivo_p2, encoding='utf-8')
        print(f"Archivo P2 cargado: {len(df_p2)} registros")
        dataframes.append(df_p2)
    except FileNotFoundError:
        print(f"No se encontró el archivo: {archivo_p2}")
    except Exception as e:
        print(f"Error al leer {archivo_p2}: {e}")
    
    if not dataframes:
        print("No se encontraron archivos para combinar.")
        return
    
    # Combinar DataFrames
    df_combinado = pd.concat(dataframes, ignore_index=True)
    print(f"Total de registros antes de eliminar duplicados: {len(df_combinado)}")
    
    # Eliminar duplicados por SKU (mantener el primero)
    df_combinado = df_combinado.drop_duplicates(subset=['SKU'], keep='first')
    print(f"Total de registros después de eliminar duplicados por SKU: {len(df_combinado)}")
    
    # Ordenar por marca y nombre
    df_combinado = df_combinado.sort_values(['Marca', 'Nombre'], na_position='last')
    
    # Guardar archivo combinado
    nombre_archivo_final = f"sears_smartphones_completo_{fecha_hoy}.csv"
    df_combinado.to_csv(nombre_archivo_final, index=False, encoding='utf-8')
    
    print(f"\nArchivo combinado creado: {nombre_archivo_final}")
    print(f"Total de productos únicos: {len(df_combinado)}")
    
    # Mostrar estadísticas
    print("\n=== ESTADÍSTICAS ===")
    print(f"Total de registros: {len(df_combinado)}")
    print(f"Registros con marca: {len(df_combinado[df_combinado['Marca'].notna() & (df_combinado['Marca'] != '')])}")
    print(f"Registros con color: {len(df_combinado[df_combinado['Color'].notna() & (df_combinado['Color'] != '')])}")
    print(f"Registros con capacidad: {len(df_combinado[df_combinado['Capacidad'].notna() & (df_combinado['Capacidad'] != '')])}")
    
    # Mostrar primeros 5 registros
    print("\n=== PRIMEROS 5 REGISTROS ===")
    print(df_combinado.head())
    
    return df_combinado

# Ejecutar la función
df_final = unir_archivos_sears_p1_p2()

Archivo P1 cargado: 500 registros
Archivo P2 cargado: 500 registros
Total de registros antes de eliminar duplicados: 1000
Total de registros después de eliminar duplicados por SKU: 1000

Archivo combinado creado: sears_smartphones_completo_2026-05-14.csv
Total de productos únicos: 1000

=== ESTADÍSTICAS ===
Total de registros: 1000
Registros con marca: 986
Registros con color: 909
Registros con capacidad: 938

=== PRIMEROS 5 REGISTROS ===
     Marca                                             Nombre   Color  \
961  AMBAR    Ambar 3G Mobify Tapa Tapita Folder Flip Celular     NaN   
1    APPLE  APPLE IPHONE 12 PRO MAX 256GB AZUL REACONDICIO...    Azul   
228  APPLE  APPLE IPHONE 12 PRO MAX 256GB AZUL REACONDICIO...    Azul   
306  APPLE  Apple IPhone 14 Pro Max Morado 128Gb  Reacondi...  Morado   
304  APPLE  Apple IPhone 14 Pro Max Negro 128Gb  Reacondic...   Negro   

    Capacidad      SKU  SKU_URL  \
961       NaN  3364013  3364013   
1      256 GB  2460594  2460594   
228    256 GB

# URLs

In [4]:
# 🚀 EXTRACTOR DE SKUs + VENDEDOR COMPLETO PARA CSV
# ============================================================================
# Misma lógica que la celda anterior pero además extrae el vendedor (seller)

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import os
import json
from urllib.parse import urlparse

class FinalSearsSKUAndSellerExtractor:
    """Extractor de SKUs y Vendedor de Sears"""
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
            'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1'
        })
        self.results = []
    
    def extract_from_url(self, url: str) -> dict:
        """Extrae el SKU y el vendedor de una URL de Sears"""
        try:
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            
            sku = self._find_sku_pattern(soup)
            product_title = self._get_product_title(soup)
            seller = self._find_seller(soup, response.text)
            
            return {
                'url': url,
                'product_id_from_url': self._extract_id_from_url(url),
                'actual_sku': sku,
                'seller': seller,
                'product_title': product_title,
                'status': 'success' if sku else 'sku_not_found'
            }
            
        except Exception as e:
            return {
                'url': url,
                'product_id_from_url': self._extract_id_from_url(url),
                'actual_sku': None,
                'seller': None,
                'product_title': None,
                'status': f'error: {str(e)}'
            }
    
    # =========================================================================
    # EXTRACCIÓN DE VENDEDOR (SELLER)
    # =========================================================================
    def _find_seller(self, soup: BeautifulSoup, raw_html: str) -> str:
        """Busca el vendedor/seller en la página usando múltiples estrategias"""
        
        # 1. PRIORITARIO: Buscar en el componente específico de Sears
        seller = self._find_seller_sears_component(soup)
        if seller:
            return seller
        
        # 2. Buscar en JSON-LD (datos estructurados)
        seller = self._find_seller_in_json_ld(soup)
        if seller:
            return seller
        
        # 3. Buscar en scripts JavaScript (variables JS del producto)
        seller = self._find_seller_in_scripts(soup, raw_html)
        if seller:
            return seller
        
        # 4. Buscar en elementos del DOM con selectores CSS
        seller = self._find_seller_in_dom(soup)
        if seller:
            return seller
        
        # 5. Buscar con patrones de texto en la página
        seller = self._find_seller_in_text(soup)
        if seller:
            return seller
        
        return None
    
    def _find_seller_sears_component(self, soup: BeautifulSoup) -> str:
        """Busca el vendedor en el componente específico de Sears (stylesShopData)"""
        
        # Estrategia 1: Buscar el div contenedor con clase que contiene "contenedorVendidoPor"
        contenedor = soup.find('div', class_=re.compile(r'contenedorVendidoPor', re.IGNORECASE))
        if contenedor:
            # El primer <p> con clase "vendidoPor" (sin el "1") tiene el vendedor
            vendido_por_p = contenedor.find('p', class_=re.compile(r'vendidoPor(?!1)', re.IGNORECASE))
            if vendido_por_p:
                text = vendido_por_p.get_text(strip=True)
                # Limpiar "Vendido por" del texto
                seller = re.sub(r'^Vendido\s+por\s*', '', text, flags=re.IGNORECASE).strip()
                if seller and len(seller) > 1:
                    return seller
        
        # Estrategia 2: Buscar directamente elementos <p> con clase "vendidoPor"
        vendido_elements = soup.find_all('p', class_=re.compile(r'vendidoPor(?!1)', re.IGNORECASE))
        for elem in vendido_elements:
            text = elem.get_text(strip=True)
            seller = re.sub(r'^Vendido\s+por\s*', '', text, flags=re.IGNORECASE).strip()
            if seller and len(seller) > 1:
                return seller
        
        # Estrategia 3: Buscar con selectores parciales de clase de Sears
        sears_selectors = [
            '[class*="ShopData"]',
            '[class*="shopData"]',
            '[class*="vendidoPor"]',
            '[class*="VendidoPor"]',
        ]
        for selector in sears_selectors:
            try:
                elements = soup.select(selector)
                for elem in elements:
                    text = elem.get_text(strip=True)
                    if 'vendido por' in text.lower():
                        seller = re.sub(r'^.*?Vendido\s+por\s*', '', text, flags=re.IGNORECASE).strip()
                        # Si queda texto con SKU pegado, cortar en "SKU"
                        seller = re.split(r'SKU\s*', seller, flags=re.IGNORECASE)[0].strip()
                        if seller and len(seller) > 1:
                            return seller
            except:
                continue
        
        return None
    
    def _find_seller_in_json_ld(self, soup: BeautifulSoup) -> str:
        """Busca el vendedor en datos JSON-LD"""
        scripts = soup.find_all('script', type='application/ld+json')
        for script in scripts:
            try:
                data = json.loads(script.string)
                seller = self._extract_seller_from_json(data)
                if seller:
                    return seller
            except:
                continue
        return None
    
    def _extract_seller_from_json(self, data) -> str:
        """Extrae vendedor recursivamente de datos JSON"""
        if isinstance(data, dict):
            # Campos comunes de vendedor en JSON-LD
            seller_fields = [
                'seller', 'vendor', 'merchant', 'soldBy',
                'offeredBy', 'provider', 'brand'
            ]
            for field in seller_fields:
                if field in data:
                    val = data[field]
                    if isinstance(val, dict):
                        name = val.get('name') or val.get('legalName') or val.get('alternateName')
                        if name and isinstance(name, str) and len(name.strip()) > 1:
                            return name.strip()
                    elif isinstance(val, str) and len(val.strip()) > 1:
                        return val.strip()
            
            # Buscar en offers -> seller
            if 'offers' in data:
                offers = data['offers']
                if isinstance(offers, dict):
                    seller = self._extract_seller_from_json(offers)
                    if seller:
                        return seller
                elif isinstance(offers, list):
                    for offer in offers:
                        seller = self._extract_seller_from_json(offer)
                        if seller:
                            return seller
            
            # Buscar recursivamente
            for key, value in data.items():
                if isinstance(value, (dict, list)) and key not in seller_fields:
                    seller = self._extract_seller_from_json(value)
                    if seller:
                        return seller
        
        elif isinstance(data, list):
            for item in data:
                seller = self._extract_seller_from_json(item)
                if seller:
                    return seller
        
        return None
    
    def _find_seller_in_scripts(self, soup: BeautifulSoup, raw_html: str) -> str:
        """Busca el vendedor en scripts JavaScript"""
        # Patrones comunes en variables JS de ecommerce
        js_patterns = [
            r'"seller"\s*:\s*"([^"]+)"',
            r'"sellerName"\s*:\s*"([^"]+)"',
            r'"vendor"\s*:\s*"([^"]+)"',
            r'"vendorName"\s*:\s*"([^"]+)"',
            r'"soldBy"\s*:\s*"([^"]+)"',
            r'"merchant"\s*:\s*"([^"]+)"',
            r'"merchantName"\s*:\s*"([^"]+)"',
            r'"storeName"\s*:\s*"([^"]+)"',
            r'"shopName"\s*:\s*"([^"]+)"',
            r"'seller'\s*:\s*'([^']+)'",
            r"'sellerName'\s*:\s*'([^']+)'",
            r"'vendor'\s*:\s*'([^']+)'",
            r'"seller_name"\s*:\s*"([^"]+)"',
            r'"vendedor"\s*:\s*"([^"]+)"',
        ]
        
        for pattern in js_patterns:
            match = re.search(pattern, raw_html, re.IGNORECASE)
            if match:
                seller = match.group(1).strip()
                if seller and len(seller) > 1 and seller.lower() not in ['null', 'undefined', 'none', '']:
                    return seller
        
        return None
    
    def _find_seller_in_dom(self, soup: BeautifulSoup) -> str:
        """Busca el vendedor en elementos del DOM"""
        # Selectores CSS comunes para vendedor
        seller_selectors = [
            # Clases e IDs comunes
            '[class*="seller"]', '[class*="Seller"]', '[class*="SELLER"]',
            '[class*="vendor"]', '[class*="Vendor"]',
            '[class*="merchant"]', '[class*="Merchant"]',
            '[class*="sold-by"]', '[class*="soldBy"]', '[class*="SoldBy"]',
            '[class*="vendedor"]', '[class*="Vendedor"]',
            '[class*="tienda"]', '[class*="Tienda"]',
            '[class*="store-name"]', '[class*="storeName"]',
            '[id*="seller"]', '[id*="vendor"]', '[id*="merchant"]',
            '[id*="soldBy"]', '[id*="vendedor"]',
            # Data attributes
            '[data-seller]', '[data-vendor]', '[data-merchant]',
            '[data-sold-by]', '[data-seller-name]',
        ]
        
        for selector in seller_selectors:
            try:
                elements = soup.select(selector)
                for element in elements:
                    # Verificar data attributes primero
                    for attr in ['data-seller', 'data-vendor', 'data-merchant', 
                                 'data-sold-by', 'data-seller-name']:
                        val = element.get(attr, '').strip()
                        if val and len(val) > 1:
                            return val
                    
                    # Verificar texto del elemento
                    text = element.get_text(strip=True)
                    if text and len(text) > 1 and len(text) < 100:
                        # Limpiar prefijos comunes
                        cleaned = re.sub(r'^(Vendido por|Sold by|Vendedor|Seller|Tienda|Proveedor)\s*:?\s*', 
                                        '', text, flags=re.IGNORECASE).strip()
                        if cleaned and len(cleaned) > 1:
                            return cleaned
            except:
                continue
        
        return None
    
    def _find_seller_in_text(self, soup: BeautifulSoup) -> str:
        """Busca el vendedor con patrones de texto en la página"""
        page_text = soup.get_text()
        
        text_patterns = [
            r'Vendido\s+por\s*:?\s*([^\n,\.]{2,50})',
            r'Vendedor\s*:?\s*([^\n,\.]{2,50})',
            r'Sold\s+by\s*:?\s*([^\n,\.]{2,50})',
            r'Seller\s*:?\s*([^\n,\.]{2,50})',
            r'Tienda\s*:?\s*([^\n,\.]{2,50})',
            r'Proveedor\s*:?\s*([^\n,\.]{2,50})',
            r'Distribuido\s+por\s*:?\s*([^\n,\.]{2,50})',
            r'Enviado\s+por\s*:?\s*([^\n,\.]{2,50})',
        ]
        
        for pattern in text_patterns:
            match = re.search(pattern, page_text, re.IGNORECASE)
            if match:
                seller = match.group(1).strip()
                # Filtrar resultados que son muy genéricos o basura
                noise_words = ['sears', 'precio', 'envío', 'gratis', 'producto', 
                              'descripción', 'características', 'opiniones']
                if (seller and len(seller) > 1 and 
                    seller.lower() not in noise_words and
                    not seller.startswith('$') and
                    not seller.isdigit()):
                    return seller
        
        return None
    
    # =========================================================================
    # EXTRACCIÓN DE SKU (misma lógica de la celda anterior)
    # =========================================================================
    def _find_sku_pattern(self, soup: BeautifulSoup) -> str:
        """Busca el patrón 'SKU ALFANUMERICO' en la página"""
        page_text = soup.get_text()
        
        sku_patterns = [
            r'SKU\s+(\d{10,20})(?=Marca|MARCA|Brand|BRAND|Model|MODELO)',
            r'SKU:\s*(\d{10,20})(?=Marca|MARCA|Brand|BRAND|Model|MODELO)',
            r'SKU\s*(\d{10,20})(?=Marca|MARCA|Brand|BRAND|Model|MODELO)',
            r'Código\s+(\d{10,20})(?=Marca|MARCA|Brand|BRAND|Model|MODELO)',
            r'Código:\s*(\d{10,20})(?=Marca|MARCA|Brand|BRAND|Model|MODELO)',
            r'(\d{12,20})(?=[A-Z][a-z])',
            r'(?:^|\s)(\d{10,20})(?=[A-Z]{2,})',
            r'SKU\s*:?\s*(\d{8,20})(?=\s|[A-Z]|$)',
            r'(?:SKU|Código|Item|Modelo)\s*:?\s*(\d{10,20})(?=\s|[A-Z]|$)',
            r'SKU\s+([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'SKU:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'SKU\s*:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'SKU\s*-\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'SKU\s*#\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'SKU\s*№\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'SKU\s+([A-Za-z]+(?:\s+[A-Za-z0-9]+){1,3})(?:\s+[A-Z]{2,}|\s*$)',
            r'SKU:\s*([A-Za-z]+(?:\s+[A-Za-z0-9]+){1,3})(?:\s+[A-Z]{2,}|\s*$)',
            r'Código\s*:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'Código\s+([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'Modelo\s*:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'Modelo\s+([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'Item\s*:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'Item\s+([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'Referencia\s*:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'Part\s*Number\s*:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'P/N\s*:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
            r'MPN\s*:\s*([A-Za-z0-9\-_/\.]+)(?:\s|$|[^A-Za-z0-9\-_/\.])',
        ]
        
        for i, pattern in enumerate(sku_patterns):
            matches = re.findall(pattern, page_text, re.IGNORECASE)
            if matches:
                for match in matches:
                    cleaned_match = self._clean_sku(match.strip())
                    if (cleaned_match and 
                        (re.search(r'[A-Za-z]', cleaned_match) or 
                        (cleaned_match.isdigit() and len(cleaned_match) >= 4))):
                        if i < 7:
                            return cleaned_match
                        return cleaned_match
        
        # Buscar en elementos del DOM
        product_info_selectors = [
            '.product-details', '.product-info', '.product-specs',
            '.specification', '.details', '.info', '.product-meta',
            '[class*="sku"]', '[class*="SKU"]', '[class*="codigo"]',
            '[id*="sku"]', '[id*="SKU"]', '[id*="codigo"]',
            '.product-code', '.item-code', '.model-code',
            '[data-sku]', '[data-product-sku]', '[data-code]'
        ]
        
        for selector in product_info_selectors:
            elements = soup.select(selector)
            for element in elements:
                element_text = element.get_text()
                for i, pattern in enumerate(sku_patterns):
                    matches = re.findall(pattern, element_text, re.IGNORECASE)
                    if matches:
                        for match in matches:
                            cleaned_match = self._clean_sku(match.strip())
                            if (cleaned_match and 
                                (re.search(r'[A-Za-z]', cleaned_match) or 
                                (cleaned_match.isdigit() and len(cleaned_match) >= 4))):
                                if i < 7:
                                    return cleaned_match
                                return cleaned_match
        
        # Buscar en atributos de datos
        data_attributes = soup.find_all(attrs={'data-sku': True})
        for elem in data_attributes:
            sku_value = elem.get('data-sku', '').strip()
            if sku_value and len(sku_value) >= 3:
                cleaned_sku = self._clean_sku(sku_value)
                if cleaned_sku:
                    return cleaned_sku
        
        # Buscar en scripts JSON
        scripts = soup.find_all('script', type='application/ld+json')
        for script in scripts:
            try:
                data = json.loads(script.string)
                sku = self._extract_sku_from_json(data)
                if sku:
                    cleaned_sku = self._clean_sku(sku)
                    if cleaned_sku:
                        return cleaned_sku
            except:
                continue
        
        return None
    
    def _clean_sku(self, sku_text: str) -> str:
        """Limpia el SKU removiendo palabras no deseadas"""
        if not sku_text:
            return None
        
        words_to_remove = [
            'marca', 'brand', 'modelo', 'model', 'color', 'colour',
            'nuevo', 'new', 'sellado', 'sealed', 'libre', 'free',
            'fabrica', 'factory', 'original', 'genuine', 'authentic',
            'oficial', 'official', 'garantia', 'warranty', 'year',
            'años', 'meses', 'months', 'dias', 'days', 'gb', 'tb',
            'negro', 'black', 'blanco', 'white', 'azul', 'blue',
            'rojo', 'red', 'verde', 'green', 'rosa', 'pink',
            'dorado', 'gold', 'plata', 'silver', 'gris', 'gray',
            'purple', 'morado', 'amarillo', 'yellow', 'naranja', 'orange',
            'reacondicionado', 'refurbished', 'usado', 'used',
            'appleid', 'apple', 'samsung', 'huawei', 'xiaomi', 'lg'
        ]
        
        cleaned = sku_text.strip()
        
        # Eliminar "Marca" y todo lo que viene después
        cleaned = re.sub(r'Marca.*', '', cleaned)
        cleaned = re.sub(r'MARCA.*', '', cleaned)
        cleaned = re.sub(r'marca.*', '', cleaned)
        cleaned = re.sub(r'Brand.*', '', cleaned)
        cleaned = re.sub(r'BRAND.*', '', cleaned)
        cleaned = re.sub(r'brand.*', '', cleaned)
        cleaned = re.sub(r'\s+Marca.*', '', cleaned)
        cleaned = re.sub(r'\s+MARCA.*', '', cleaned)
        cleaned = re.sub(r'\s+marca.*', '', cleaned)
        cleaned = re.sub(r'\s+Brand.*', '', cleaned)
        cleaned = re.sub(r'\s+BRAND.*', '', cleaned)
        cleaned = re.sub(r'\s+brand.*', '', cleaned)
        cleaned = cleaned.strip()
        
        # Si es número largo pegado a letras, extraer solo la parte numérica
        if re.match(r'^\d{8,}[A-Za-z]', cleaned):
            numeric_match = re.match(r'^(\d{8,})', cleaned)
            if numeric_match:
                cleaned = numeric_match.group(1)
        
        # Remover palabras no deseadas separadas por espacios
        for word in words_to_remove:
            cleaned = re.sub(rf'^{re.escape(word)}\s+', '', cleaned, flags=re.IGNORECASE)
            cleaned = re.sub(rf'\s+{re.escape(word)}$', '', cleaned, flags=re.IGNORECASE)
            cleaned = re.sub(rf'\s+{re.escape(word)}\s+', ' ', cleaned, flags=re.IGNORECASE)
        
        cleaned = re.sub(r'^\s+|\s+$', '', cleaned)
        cleaned = re.sub(r'\s+', ' ', cleaned)
        
        if len(cleaned) < 3:
            return None
        if cleaned.replace(' ', '').isdigit() and len(cleaned.replace(' ', '')) < 4:
            return None
        if not re.search(r'[A-Za-z0-9]', cleaned):
            return None
        if len(cleaned) > 40:
            return None
        if cleaned.strip() == '':
            return None
        
        return cleaned if cleaned else None
    
    def _extract_sku_from_json(self, data) -> str:
        """Extrae SKU de datos JSON estructurados"""
        if isinstance(data, dict):
            sku_fields = ['sku', 'SKU', 'productID', 'itemId', 'identifier', 
                         'mpn', 'model', 'gtin', 'productCode', 'itemCode']
            for field in sku_fields:
                if field in data and data[field]:
                    sku_value = str(data[field]).strip()
                    if len(sku_value) >= 3:
                        cleaned_sku = self._clean_sku(sku_value)
                        if cleaned_sku:
                            return cleaned_sku
            for key, value in data.items():
                if isinstance(value, (dict, list)):
                    sku = self._extract_sku_from_json(value)
                    if sku:
                        return sku
        elif isinstance(data, list):
            for item in data:
                sku = self._extract_sku_from_json(item)
                if sku:
                    return sku
        return None
    
    def _extract_id_from_url(self, url: str) -> str:
        """Extrae el ID del producto de la URL"""
        match = re.search(r'/producto/(\d+)/', url)
        return match.group(1) if match else None
    
    def _get_product_title(self, soup: BeautifulSoup) -> str:
        """Extrae el título del producto"""
        selectors = ['h1', '.product-title', '.product-name', 
                     '[data-testid="product-title"]', '.ProductTitle']
        for selector in selectors:
            element = soup.select_one(selector)
            if element and element.get_text().strip():
                return element.get_text().strip()[:100]
        return "Título no encontrado"
    
    def process_urls_from_csv(self, csv_file_path: str, delay: float = 1.5) -> list:
        """Procesa todas las URLs desde un archivo CSV"""
        urls = self._load_urls_from_csv(csv_file_path)
        if not urls:
            print("❌ No se pudieron cargar URLs del archivo CSV")
            return []
        
        valid_urls = [url for url in urls if self._validate_sears_url(url)]
        if not valid_urls:
            print("❌ No se encontraron URLs válidas de Sears")
            return []
        
        print(f"📊 Procesando {len(valid_urls)} URLs válidas de Sears...")
        
        results = []
        for i, url in enumerate(valid_urls, 1):
            print(f"📋 [{i}/{len(valid_urls)}] Procesando: {url[:70]}...")
            
            result = self.extract_from_url(url)
            results.append(result)
            
            sku_str = result['actual_sku'] or '—'
            seller_str = result['seller'] or '—'
            if result['actual_sku']:
                print(f"  ✅ SKU: {sku_str}  |  🏪 Vendedor: {seller_str}")
            else:
                print(f"  ❌ SKU no encontrado  |  🏪 Vendedor: {seller_str}")
            
            if i < len(valid_urls):
                time.sleep(delay)
        
        self.results = results
        return results
    
    def _load_urls_from_csv(self, csv_file_path: str) -> list:
        """Carga URLs desde un archivo CSV"""
        try:
            try:
                df = pd.read_csv(csv_file_path, header=None)
                urls = df.iloc[:, 0].dropna().tolist()
            except:
                with open(csv_file_path, 'r', encoding='utf-8') as file:
                    urls = [line.strip() for line in file if line.strip() and not line.startswith('#')]
            urls = [url for url in urls if isinstance(url, str) and url.startswith('http')]
            print(f"✅ Cargadas {len(urls)} URLs desde {csv_file_path}")
            return urls
        except Exception as e:
            print(f"❌ Error cargando CSV: {e}")
            return []
    
    def _validate_sears_url(self, url: str) -> bool:
        """Valida si una URL pertenece a Sears"""
        sears_domains = ['sears.com', 'sears.com.mx', 'www.sears.com', 'www.sears.com.mx']
        try:
            parsed_url = urlparse(url)
            return parsed_url.netloc.lower() in sears_domains
        except:
            return False
    
    def save_to_excel(self, filename: str = "sears_skus_vendedor.xlsx"):
        """Guarda los resultados en Excel"""
        if not self.results:
            print("❌ No hay resultados para guardar")
            return
        
        df = pd.DataFrame(self.results)
        column_order = ['url', 'product_id_from_url', 'actual_sku', 'seller', 'product_title', 'status']
        df = df.reindex(columns=column_order)
        
        df.to_excel(filename, index=False, engine='openpyxl')
        print(f"💾 Resultados guardados en: {filename}")
        
        # Estadísticas
        successful = len(df[df['status'] == 'success'])
        sellers_found = len(df[df['seller'].notna()])
        total = len(df)
        success_rate = (successful / total) * 100 if total > 0 else 0
        seller_rate = (sellers_found / total) * 100 if total > 0 else 0
        
        print(f"📊 ESTADÍSTICAS FINALES:")
        print(f"  ✅ Total procesadas: {total}")
        print(f"  ✅ SKUs extraídos: {successful} ({success_rate:.1f}%)")
        print(f"  🏪 Vendedores encontrados: {sellers_found} ({seller_rate:.1f}%)")
        print(f"  ❌ Fallos SKU: {total - successful}")
        
        return df

# ============================================================================
# CONFIGURACIÓN Y EJECUCIÓN
# ============================================================================

CSV_FILE_PATH = r"c:\Users\c50012218\Downloads\urls sears.csv"
OUTPUT_FILE = "sears_skus_VENDEDOR.xlsx"
DELAY_SECONDS = 1.5

print("🚀 EXTRACTOR DE SKUs + VENDEDOR DE SEARS")
print("=" * 50)
print(f"📁 Archivo CSV: {CSV_FILE_PATH}")
print(f"💾 Archivo de salida: {OUTPUT_FILE}")
print(f"⏱️ Delay entre solicitudes: {DELAY_SECONDS} segundos")

if not os.path.exists(CSV_FILE_PATH):
    print(f"\n❌ ERROR: No se encontró el archivo CSV")
    print(f"📍 Ruta buscada: {CSV_FILE_PATH}")
    print("💡 Modifica la variable CSV_FILE_PATH con la ruta correcta")
else:
    print(f"\n✅ Archivo CSV encontrado")
    
    extractor = FinalSearsSKUAndSellerExtractor()
    
    print(f"\n🚀 Iniciando procesamiento completo...")
    results = extractor.process_urls_from_csv(CSV_FILE_PATH, DELAY_SECONDS)
    
    if results:
        df_result = extractor.save_to_excel(OUTPUT_FILE)
        
        successful_results = df_result[df_result['status'] == 'success']
        if len(successful_results) > 0:
            print(f"\n📋 MUESTRA DE RESULTADOS:")
            for i, row in successful_results.head(10).iterrows():
                seller_display = row['seller'] if pd.notna(row['seller']) else '—'
                print(f"  • ID: {row['product_id_from_url']} → SKU: {row['actual_sku']} | 🏪 {seller_display}")
            
            if len(successful_results) > 10:
                print(f"  ... y {len(successful_results) - 10} más")
        
        print(f"\n🎉 ¡PROCESAMIENTO COMPLETADO!")
        print(f"📋 Revisa el archivo: {OUTPUT_FILE}")
    else:
        print("\n❌ No se pudieron procesar las URLs")

🚀 EXTRACTOR DE SKUs + VENDEDOR DE SEARS
📁 Archivo CSV: c:\Users\c50012218\Downloads\urls sears.csv
💾 Archivo de salida: sears_skus_VENDEDOR.xlsx
⏱️ Delay entre solicitudes: 1.5 segundos

✅ Archivo CSV encontrado

🚀 Iniciando procesamiento completo...
✅ Cargadas 13 URLs desde c:\Users\c50012218\Downloads\urls sears.csv
📊 Procesando 13 URLs válidas de Sears...
📋 [1/13] Procesando: https://www.sears.com.mx/producto/3313428/apple-iphone-15-plus-256gb-n...
  ✅ SKU: 71769227398229  |  🏪 Vendedor: Reuse Mexico
📋 [2/13] Procesando: https://www.sears.com.mx/producto/3981258/celular-vivo-v70-fe-512gb-5g...
  ✅ SKU: 2266602  |  🏪 Vendedor: SEARS
📋 [3/13] Procesando: https://www.sears.com.mx/producto/3977455/google-pixel-8-128-gb-verde-...
  ✅ SKU: GPIXV128  |  🏪 Vendedor: IDESAIN.MX
📋 [4/13] Procesando: https://www.sears.com.mx/producto/3978581/samsung-galaxy-a37-5g-256gb-...
  ✅ SKU: A37MRD2564  |  🏪 Vendedor: Solucion Celular
📋 [5/13] Procesando: https://www.sears.com.mx/producto/3978634/samsun